***
# AirBnB Listings in Zurich: Data analysis
***
Before delving into the analysis of Airbnb listings in the city of Zurich, let's iinstall all necessary libraries:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import geopandas as gpd
import folium
import re

***
## 1. About the Project
### 1.1. AirBnB listings as a topic
The short-term rental market, AirBnB in particular, has grown rapidly in urban areas, influencing rents, local economies, and urban planning. In cities like Zürich, understanding the factors of Airbnb prices can provide insights into market dynamics, potential regulatory interventions, and correlations with traditional rental prices. 
### 1.2. The Datasets
Inside AirBnB, the provider of our main dataset, is a non-commercial third-party provider of AirBnB data that aims to increase transparency of Airbnb activity. They scrape the official Airbnb website regularly and structure it csv files which can be obtained via their website https://insideairbnb.com/.
In addition, we are using two other datasets, one for normalizing the airbnb listings over neighborhoods in Zurich and the other to compare these normalized listing quantities with rental prices in each neighborhood.
### 1.3. Our Goals
In our project we aim to analyse Airbnb listings in Zürich, identify key features influencing pricing, and compare them with municipal rental statistics.
***
## 2. Loading, Cleaning and Validating the datasets
### 2.1. Loading and displaying dataframe overviews


In [ ]:
# define an overview function
def overview(df, name="dataset"):
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True)
    }).sort_values(["dtype", "missing_pct"], ascending=[True, False])

    print(f"\n=== {name} ===")
    print("Shape:", df.shape)
    return summary

#loading and displaying
## Airbnb
listings_path = 'data/listings.csv'
airbnb_df= pd.read_csv(listings_path, encoding="utf-8")
print(overview(airbnb_df, "Airbnb"))

## housing stock
housing_path = 'data/bau522od5221_wohnungsbestand_zurich.csv'
housing_df = pd.read_csv(housing_path, encoding="utf-8")
display(overview(housing_df, "Housing"))

## Rental prices
rental_path = 'data/rental_prices.csv'
rental_df = pd.read_csv(rental_path, encoding="utf-8")
display(overview(rental_df,"Rental"))

### 2.2. Cleaning
Based on the overview, we have decided to clean the datasets the following way:
- Generally:
    - Standardize column names (lowercase, underscores etc.).
    - Turn every column with 2 unique values to boolean.
    - All object data types are converted to more primitive dtypes (string, float, int, category, sorted category).
- Airbnb:
    - Drop unnecessary columns (i.e. everything with a URL).
    - Remove columns with too much unusable noise, like *host_description* in *airbnb_df*. Instead, we added a column that states the length of the entry.
    - Reference all dates or temporal variables to the date the data was scraped.
    - Feature engineering amenities in teh Airbnb dataframe by extracting amenities from the list in the *amenities* column.
- Housing Stock:
    - Only keep entries from 2025
    - Restructure the dataframe to only have 32 rows(one for each neighborhood). Aggregate the number of objects into separate columns for different apartment sizes.
- Rental Prices:
    - Only keep entries for 2024 (remove 2022)
    - Only keep square meter entries
    - Only keep netto entries (to remove differences in accounting etc. between neighborhoods as much as possible)
    - Restructure the dataframe to only have 32 rows (one for each neighborhood). The columns now represent different rental prices.

A unique normalization function was defined for each of the 3 datasets in the file *normalize.py*. Consult this file for further detail on the normalization process.

In [ ]:
from src.normalize import normalize_airbnb, normalize_housing, normalize_rental

# normalize Airbnb df
airbnb_df_norm = normalize_airbnb(airbnb_df)

# normalize housing stock df
housing_df_norm = normalize_housing(housing_df, year=2025)

# normalize rental price df
rental_df_norm = normalize_rental(rental_df, year=2024, brutto=False, sqm=True, level = 5, cat_zimmer=False)

***
## 3. Aggregating to quartier-level
At this point, the housing stock dataframe and rental price dataframe contain hundreds of rows, where each Quartier has several entries. For our analysis on quartier-level, we prefer a different structure, where each quartier only has one row. For that reson, we create a new pivot table around the quartier-column.
### 3.1. Using Pivot Tables

In [ ]:
from src.aggregation import quartiere_standardized_housing, quartiere_standardized_rental

# housing stock
housing_df_quart = quartiere_standardized_housing(housing_df_norm)
# flatten columns of housing df to 1:
housing_df_quart.columns = [
    "_".join(col).strip() if isinstance(col, tuple) else col
    for col in housing_df_quart.columns
]
housing_df_quart = housing_df_quart.drop(columns="index_")
display(housing_df_quart.head())

# rental prices
rental_df_quart = quartiere_standardized_rental(rental_df_norm)
display(rental_df_quart.head())

Now, the two datasets are of length 35 (i.e. both only contain one row for each of the 34 quartiere).
### 3.2. Add Airbnb count to Housing Stock dataframe
In order to compare airbnb listings (normalized by housing stock) and rental prices in each quartier, we need to know how many listings are in each quartier. Luckily, INside Airbnb provides the quartier that each listing lies in, so we simply need to count them and merge that count to *housing_df_quart*:

In [ ]:
# count airbnbs per quartier
counts_airbnb_quartier = (
    airbnb_df_norm
    .groupby("neighbourhood_cleansed")
    .size()
    .rename("n_airbnbs")
)

# merge
housing_df_quart = housing_df_quart.merge(
    counts_airbnb_quartier,
    left_on="quarlang_",
    right_index=True,
    how="left"
)


***
## 4. Exploratory Data Analysis
### 4.1. Airbnb Dataset
#### 4.1.1. Price
Let's simply calculate the minimum and maximum price for a listing in the dataset:

In [ ]:
minimum = min(airbnb_df_norm["price"])
maximum = max(airbnb_df_norm["price"])
print(f"Airbnb prices:\nMin: {minimum}\nMax: {maximum}")

Now, let us take a look at the distribution of listing prices in the dataset. We generate a histogram and a boxplot. Since there are some heavy outliers with very high prices, let us also crop the two plots to inspect the distribution at 'normal' price ranges (between 0 and 1000).

In [ ]:
print(max(airbnb_df_norm["price"]))

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize = (10,7))

ax1.hist(airbnb_df_norm["price"], bins = 100)
ax1.set_xlabel("price")
ax1.set_ylabel("count")
ax1.set_title("Histogram 'price' (bin=100)")

sns.boxplot(ax=ax2, data=airbnb_df_norm, y="price")
ax2.set_title("Boxplot 'price'")

ax3.hist(airbnb_df_norm["price"], bins = 1000)
ax3.set_xlim(0,1000)
ax3.set_xlabel("price")
ax3.set_ylabel("count")
ax3.set_title("Histogram 'price' cropped (bin=1000)")

sns.boxplot(ax=ax4, data=airbnb_df_norm, y="price")
ax4.set_ylim(0,500)
ax4.set_title("Boxplot 'price' cropped")

plt.tight_layout()
plt.show()

These plots show the following:
- Positive skew: mostly prices between 0 and 500, but outliers all the way at 10000
    - i.e. many cheap and moderately priced listings with few very expensive listings.
    - This means that we should be looking at the median rather than the mean when trying to get one metric for price.
    - These outliers are most likely real listings, since a market for luxury airbnb listings does exist.
- There is a singular peak: no submarkets visible (e.g. high demand for luxury listings could have resultet in a small peak at higher prices).
- 25th and 75th quantile lie at roughly 100 and 200 CHF respectively.
#### 4.1.2. Reviews
There are 7 variables for different review scores. Since we aim to use some of them in our analysis later on, it would be helpful to see how their values are structures.

In [ ]:
cols = ["review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness", "review_scores_checkin", "review_scores_communication", "review_scores_location", "review_scores_value"]

fig, ax = plt.subplots(figsize=(12,4))
sns.boxplot(ax=ax, data=airbnb_df_norm[cols],
            order=["review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness", "review_scores_checkin", "review_scores_communication", "review_scores_location", "review_scores_value"]
            )
ax.set_xticks(range(7))
ax.set_xticklabels(["Overall", "Accuracy", "Cleanliness", "Checkin", "Communication", "Location", "Value"])
ax.set_ylim(1,5)
plt.tight_layout()
plt.show()

As expected, all review categories are mostly positive, with medians around 4.7. Of course, there are also outliers toward the bottom, as can be expected when asking for user feedback. We are especially interested in the last variable: Value since we will try to predict those values. We can see, that it has by far the lowest mean and one of the largest variances. We can tell by this shape, that it is heavily left-skewed. We will probably have to transform it in order to use it in a regression.
#### 4.1.3. Geographic distribution
Since each listing contains values for longitude and latitude, we can plot them on a map to get a sense of how they are distributed geographically across the city. We will add the borders of the quartiere aswell as colour the listings according to their corresponding quartier.  
For this process, we need to temporarily transform our airbnb listings dataframe into a geodataframe using geopandas:

In [ ]:
# convert airbnb_df_norm to geo data frame
airbnb_gdf = gpd.GeoDataFrame(
    airbnb_df_norm,
    geometry=gpd.points_from_xy(airbnb_df_norm["longitude"], airbnb_df_norm["latitude"])).set_crs(epsg=4326)

# import quartiere borders and forested areas
filepath_quart = "data/zurich_quartiere.gpkg"
filepath_for = "data/forest_zurich.gpkg"
layer_quart = "stzh.adm_statistische_quartiere_map"
quartiere = gpd.read_file(filepath_quart, layer=layer_quart).to_crs(epsg=4326)
forest = gpd.read_file(filepath_for).to_crs(epsg=4326)

# create map with function from src/maps.py
from src.maps import zurich_map_eda
zurich_map = zurich_map_eda(airbnb_gdf, forest, quartiere)
zurich_map

By **hovering over the empty land in the quartier**, the name of the quartier shows up. The color ramp uses the log-transform of the price, so the legend is rather vague. We mainly use the colors to convey the general distribution of listing prices. By hovering over a listing, one can see the actual price which should give an idea of how the color ramp works.
We can see that airbnb **listings are clustered along the lake shore and in the city center**. Forested areas obviously do not contain any listings, but listings reach the edge basically anywhere else.  
Concerning the **price**, we can see that prices are generally **higher in the city center and lower at the municipalitites borders**, although clusters of high prices can be found outside the center like on the lake shores or in Höngg. This price distribution will become important later on when we will try to train a model which will **predict listing prices based on a set of parameters**. Looking at this map, we guess that **location could play a major part in that model**.
#### 4.1.4. Correlation of dataset variables
Since we will be using several variables from the Airbnb dataset to predict things, it could be useful to see how these variables correlate with each other. All boolean variables have been converted back to 1s and 0s for this part. Then, we only took the numeric variables for the correlation matrix and dropped a bunch of variables we think do not fit for this visualization and would simply clutter the graph.  
Aigain, we are using a function for plotting the correlation matrix which is defined in our `src` folder.

In [ ]:
from src.plots import prepare_corr_df, plot_corr_heatmap

airbnb_corr = prepare_corr_df(airbnb_df_norm)
plot_corr_heatmap(airbnb_corr)

Take-aways from this correlation plot:
- obvious correlation between similar variables like `availability_eoy`and other `availability_xxx`variables. Or all the review scores among themselves.
- `host_is_superhost` has medium correlation with quite a lot of other variables (at least compared to most other pairings).
- Surprisingly to us, `price` does not show even moderate correlation with any variable except `estimated_revenue_l365d` (latter seems to be a function of the prior anyways). Knowing this, we expect it will be hard to find a set of variables which can be used to train a reliable model to predict listing price, however we can still consult this correlation matrix to get a starting set of variables, which we think might have the highest influence on the model.
- In general, negtive correlation between variables is much less common and less extreme than positive correlation. This means that there is no variable that causes another variable to decrease significantly if itself increased.
### 4.2. Housing Stock Dataset
#### 4.2.1. Basic statistics of housing stock dataset
First, let us look at how the different stock-sizes (e.g. `_1-Zimmer`or `_5-Zimmer`) are distributed across the quartiere with boxplots:

In [ ]:
# restructure to get columns by number of rooms
from src.plots import restructure_housing_by_rooms, box_and_stacked_housing_stock
housing_rooms = restructure_housing_by_rooms(housing_df_quart)

# normalize for each quartier
housing_share = housing_rooms.div(housing_rooms.sum(axis=1), axis=0)

# plotting using src/plots.py
box_and_stacked_housing_stock(housing_share)

What we can gather from the boxplot:  
- Objects with 3 rooms are the most common. This follows typical housing structure patterns where the most common sizes are 3 or 4 rooms and they get less common towards the extremes like 1 room or 6 rooms.
- There is noticeable difference in variance (e.g. larger variance for 1 room than for 5 rooms), but these differences are not extreme.
- There are a couple of outliers which we can inspect in the stacked bar chart to the right.
- The two outliers from the boxplot for 1 room are now visible: Hochschulen and Rathaus have significantly more stock with 1 room than the others. This might be due to more student housing in this area.  
- The two outliers for 6 rooms are Hottingen and Fluntern, both quartiere lie on the slopes of the Zürichberg. It makes sense that one would find larger estates there, since they are considered more rich parts of the city.  
#### 4.2.2. Geographic distribution of airbnb stock
In order to get an idea of where the airbnb listings are situated, not on the level of each individual listing, but on the level of the quartiere, we can create a simple choropleth map. But first, we need to join the airbnb counts from `housing_df_quart` with the gpkg file containing the quartiere and their boundaries: